In [32]:
import uuid
import os
import re
import shutil
from pathlib import Path

## Configs

In [33]:
BASE_DIR = Path("./teste_bucket/profile-documents")

# Exemplo: 0003ecb1-1835-445a-9b4f-59138-2024-04-09.jpg -> 0003ecb1-1835-445a-9b4f-59138...
UUID_PATTERN = re.compile(
    r"^([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})"
)

## Methods

In [ ]:
def extrair_id_pasta(nome_arquivo: str) -> str | None:
    """Extrai o UUID correspondente ao nome da pasta."""
    match = UUID_PATTERN.match(nome_arquivo)
    if match:
        return match.group(1)

    partes = nome_arquivo.split("-")
    if len(partes) >= 5:
        possivel_id = "-".join(partes[:5])
        return possivel_id

    return None


def criar_cenario_teste():
    if BASE_DIR.exists():
        shutil.rmtree(BASE_DIR.parent)

    BASE_DIR.mkdir(parents=True, exist_ok=True)

    id_1 = "00076ac7-fbc2-42f3-9620-8cb4f19bca5e"
    (BASE_DIR / id_1).mkdir(exist_ok=True)
    (BASE_DIR / id_1 / "avatar.png").touch()

    (BASE_DIR / f"{id_1}-2024-05-15.jpg").touch()

    id_2 = "0003ecb1-1835-445a-9b4f-59138d612345"
    (BASE_DIR / f"{id_2}-2024-04-09.jpg").touch()

    print("Ambiente de teste criado com sucesso em:", BASE_DIR.resolve())
    print("Arquivos e pastas iniciais:")
    for item in sorted(BASE_DIR.iterdir()):
        print(f" - [{'PASTA' if item.is_dir() else 'ARQUIVO'}] {item.name}")


def organizar_diretorio_local(dry_run: bool = False):
    """Varre os arquivos soltos e move para suas respectivas pastas com o novo formato."""
    print(f"\nIniciando organização {'(DRY-RUN)' if dry_run else ''}...")

    itens = list(BASE_DIR.iterdir())
    arquivos_soltos = [item for item in itens if item.is_file()]

    if not arquivos_soltos:
        print("Nenhum arquivo solto encontrado para organizar.")
        return

    for arquivo in arquivos_soltos:
        pasta_alvo_nome = extrair_id_pasta(arquivo.name)

        if not pasta_alvo_nome:
            print(f"Não foi possível identificar o ID de: {arquivo.name}")
            continue

        pasta_destino = BASE_DIR / pasta_alvo_nome

        # {nome_original}_{uuid}{extensao}
        extensao = arquivo.suffix
        nome_sem_extensao = arquivo.stem
        novo_uuid = str(uuid.uuid4())
        novo_nome_arquivo = f"{nome_sem_extensao}_{novo_uuid}{extensao}"

        destino_final = pasta_destino / novo_nome_arquivo

        if dry_run:
            print(f"[SIMULAÇÃO] Mover: {arquivo.name} -> {pasta_alvo_nome}/{novo_nome_arquivo}")
        else:
            pasta_destino.mkdir(parents=True, exist_ok=True)
            shutil.move(str(arquivo), str(destino_final))
            print(f"Movido: {arquivo.name} -> {pasta_alvo_nome}/{novo_nome_arquivo}")

    print("\nOrganização finalizada!")


In [35]:
criar_cenario_teste()

Ambiente de teste criado com sucesso em: /home/enzo-tabuchi/Documentos/Instituto/GCP/teste_bucket/profile-documents
Arquivos e pastas iniciais:
 - [ARQUIVO] 0003ecb1-1835-445a-9b4f-59138d612345-2024-04-09.jpg
 - [PASTA] 00076ac7-fbc2-42f3-9620-8cb4f19bca5e
 - [ARQUIVO] 00076ac7-fbc2-42f3-9620-8cb4f19bca5e-2024-05-15.jpg
--------------------------------------------------


In [36]:
organizar_diretorio_local(dry_run=True)


Iniciando organização (DRY-RUN)...
[SIMULAÇÃO] Mover: 0003ecb1-1835-445a-9b4f-59138d612345-2024-04-09.jpg -> 0003ecb1-1835-445a-9b4f-59138d612345/0003ecb1-1835-445a-9b4f-59138d612345-2024-04-09_97ce6234-e4cb-4ad5-8feb-9f29bb16a235.jpg
[SIMULAÇÃO] Mover: 00076ac7-fbc2-42f3-9620-8cb4f19bca5e-2024-05-15.jpg -> 00076ac7-fbc2-42f3-9620-8cb4f19bca5e/00076ac7-fbc2-42f3-9620-8cb4f19bca5e-2024-05-15_cf825e0b-daa5-4890-b20b-55f62fc04e7e.jpg

Organização finalizada!


In [37]:
# organizar_diretorio_local(dry_run=False)

In [38]:
# print("\nEstrutura final resultante:")
# for item in sorted(BASE_DIR.iterdir()):
#     if item.is_dir():
#         conteudo = [f.name for f in item.iterdir()]
#         print(f" {item.name}/ ({len(conteudo)} itens: {conteudo})")